In [2]:
import os
import time
import tqdm
import random
import numpy as np
import torch
import torch.nn as nn
from utils.preprocessing_support import HDF5BrainDataset
from torch.utils.data import random_split, DataLoader
from models.ContrastiveModel import ListwiseDecoderModel,train_pairwise_epoch,train_lambdarank_epoch,initialize_weights,get_image_and_contours,pairwise_bce_loss

In [ ]:
# h5_data_dir = '/Users/yifanli/Desktop/dataset/preprocessed_data_BraTS20_h5'
# dataset = HDF5BrainDataset(h5_data_dir)
# # Define train-validation split sizes
# train_size = int(0.8 * len(dataset))
# val_size = len(dataset) - train_size

# # Use a fixed random seed for reproducibility
# generator = torch.Generator().manual_seed(42)
# train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

# # Create DataLoaders
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# for data in train_loader:
#     break
# print(data.keys())
# print(data['img'].keys())
# print(data['img']['t1'].shape)

In [3]:
BATCH_SIZE = 1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cuda = True if torch.cuda.is_available() else False
Tensor = torch.cuda.FloatTensor if cuda else torch.FloatTensor

# Initialize the dataset
h5_data_dir = 'light_h5_data'
dataset = HDF5BrainDataset(h5_data_dir)

# Define train-validation split sizes
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

# Use a fixed random seed for reproducibility
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)


# model = ListwiseDecoderModel(
#     img_channels=1024,
#     contour_dim=1152,
#     hidden_dim=256,
#     n_heads=4,
#     num_decoder_layers=3
# ).to(device)
from models.ENetModel import ENetSimple,ENet,ENet_v2
h_dim = 256
n_layers = 1


In [ ]:
model = ENet(
    img_channels=1024,
    contour_dim=1152,
    hidden_dim=h_dim,
    n_heads=4,
    num_decoder_layers=n_layers
)
# dic_pth = '/Users/yifanli/Desktop/fromLinux/TumorContour/Enet_con.pth'
# dic_pth = '/Users/yifanli/Desktop/fromLinux/TumorContour/Enet_con_rank0001.pth'
# dic_pth = '/Users/yifanli/Desktop/fromLinux/TumorContour/Enet_con_rank0111.pth'
state_dict = torch.load(dic_pth, map_location=torch.device('cpu'))
model.load_state_dict(state_dict, strict=False)
model.eval()

In [12]:
from models.ENetModel import ENet_v2
model = ENet_v2(
    img_channels=1024,
    contour_dim=1152,
    hidden_dim=h_dim,
    n_heads=4,
    num_decoder_layers=n_layers
)
dic_pth = '/Users/yifanli/Desktop/fromLinux/TumorContour/Enet_v2_con_rank0111.pth'

state_dict = torch.load(dic_pth, map_location=torch.device('cpu'))
model.load_state_dict(state_dict, strict=False)
model.eval()

<All keys matched successfully>

In [13]:
def evaluate_bce_loss(pred_scores, gt_scores, i_idx, j_idx):
    """
    Pairwise BCE ranking:
     - pred_scores: (N,)
     - gt_scores: (N,), smaller => better
     - i_idx, j_idx: (num_pairs,) random pairs
       label=1 if i < j => dist(i)<dist(j)
    """
    dist_i = gt_scores[i_idx]
    dist_j = gt_scores[j_idx]
    s_i = pred_scores[i_idx]
    s_j = pred_scores[j_idx]
    
    labels = (dist_i > dist_j).float()  # 1 if i better
    delta = s_i - s_j
  
    p_ij = torch.sigmoid(delta)  # probability i < j

    bce = -(labels * torch.log(p_ij+1e-8) + (1-labels)*torch.log(1-p_ij+1e-8))
    p_ij[p_ij>0.5] = 1
    p_ij[p_ij<=0.5] = 0   
    
#     plt.scatter(np.arange(800),p_ij)
#     plt.scatter(np.arange(800),labels)
    accuracy = 1-torch.abs(p_ij - labels).sum()/ len(p_ij)
 
    return  accuracy

#   Model Inference running test

In [6]:
test_dataset = val_loader
model.eval()
total_test_loss = 0.0
total_acc = 0.0
start_time = time.time()
num_pairs = 512

with torch.no_grad():
    for val_item in test_dataset:
        image_emb, all_g_embs, all_scores = get_image_and_contours(val_item)
        if len(image_emb.shape) == 4:
            image_emb = image_emb.unsqueeze(0)

        image_emb = image_emb.to(device)
        all_g_embs = all_g_embs.to(device)
        all_scores = all_scores.to(device)

        N_test = all_scores.shape[0]

        pair_indices_test = torch.randint(low=0, high=N_test*N_test, size=(num_pairs,), device=device)
        i_idx_test = pair_indices_test // N_test
        j_idx_test = pair_indices_test % N_test

        score_i_test = all_scores[i_idx_test]
        score_j_test = all_scores[j_idx_test]

        g_i_test = all_g_embs[i_idx_test]
        g_j_test = all_g_embs[j_idx_test]

        image_batch_test = image_emb.repeat(g_i_test.size(0), 1, 1, 1, 1)

        dist_test,_ = model(image_batch_test, g_i_test, g_j_test)
       
        break

#     distance measurement test

In [156]:
import torch.nn.functional as F

def get_target_distance(si, sj):
    """
    si, sj in {0,1} or partial in (0,1).
    Return the "desired" distance.
      1) (1,1) or (0,0) => 0
      2) (1,0) or (0,1) => 1
      3) (1,d) => (1 - d)
      4) (0,d) => d
      5) (d1,d2) => |d1 - d2|
    """
    # both absolute same
    if (si == 0.0 and sj == 0.0) or (si == 1.0 and sj == 1.0):
        return 0.0
    # both absolute mismatch
    if (si == 0.0 and sj == 1.0) or (si == 1.0 and sj == 0.0):
        return 1.0
    
    # one absolute, one partial
    if si == 1.0 and (0 < sj < 1.0):
        return 1.0 - sj
    if sj == 1.0 and (0 < si < 1.0):
        return 1.0 - si
    if si == 0.0 and (0 < sj < 1.0):
        return sj
    if sj == 0.0 and (0 < si < 1.0):
        return si
    
    # both partial => difference
    return abs(si - sj)

def custom_contrastive_loss(dist_pred, score_i, score_j,detail =False):
    """
    dist_pred: (P,) predicted distances
    score_i, score_j: (P,) each in {0,1} or partial in (0,1)
    We'll compute an MSE with the 'target distance' from get_target_distance.
    """
    device = dist_pred.device

    # build a list of target distances
    dist_targets = []
    # we must do this on CPU numpy or do it in a vectorized way on GPU
    # for simplicity, we do a loop:
    si_np = score_i.detach().cpu().numpy()
    sj_np = score_j.detach().cpu().numpy()

    for s_i, s_j in zip(si_np, sj_np):
        dist_targets.append(get_target_distance(s_i, s_j))
    
    dist_targets = torch.tensor(dist_targets, device=device, dtype=torch.float32)
    
    # MSE
    if detail :
        print('# # # # # # # # # # # # # # # # # # # # # # # # # # # #')
        print('predicted distance   ', dist_pred)
        print('groundtruth distance ', dist_targets)
        print('d1', score_i)
        print('d2', score_j)
        

    loss = F.mse_loss(dist_pred, dist_targets)
    
    return loss
def test_region(idx,jdx,model,detail = False):
    
    with torch.no_grad():
        if isinstance(idx, np.ndarray) and isinstance(jdx, np.ndarray):
            g_i_test = all_g_embs[idx]
            g_j_test = all_g_embs[jdx]
            i_score = all_scores[idx]
            j_score = all_scores[jdx]
        else:
            g_i_test = all_g_embs[idx:idx+5]
            g_j_test = all_g_embs[jdx:jdx+5]
            i_score = all_scores[idx:idx+5]
            j_score = all_scores[jdx:jdx+5]
    ranking_target = (i_score > j_score).float()
    image_batch_test = image_emb.repeat(g_i_test.size(0), 1, 1, 1, 1)
    dist_test,prob_pred = model(image_batch_test, g_i_test, g_j_test)     
    contrastive_loss = custom_contrastive_loss(dist_test,i_score , j_score ,detail =detail )
    ranking_loss_all = F.binary_cross_entropy_with_logits(prob_pred, ranking_target, reduction='none')
    
    if detail :
        print('--------------------------------------------------------')
        print('The real label is     :', ranking_target)
        print('The predicted label is:', F.sigmoid(prob_pred))
        print('# # # # # # # # # # # # # # # # # # # # # # # # # # # #')
    print('contrastvie loss {:4f}'.format(contrastive_loss.item()))
    print('ranking loss     {:4f}'.format(ranking_loss_all.mean().item()))


In [157]:
detail  = True
print('Two from WT region')
test_region(400,401,model, detail=detail)

Two from WT region
# # # # # # # # # # # # # # # # # # # # # # # # # # # #
predicted distance    tensor([0.1318, 0.1100, 0.0526, 0.1033, 0.1416], grad_fn=<SqueezeBackward1>)
groundtruth distance  tensor([0.0802, 0.0773, 0.0693, 0.0627, 0.0551])
d1 tensor([0.9085, 0.8283, 0.7510, 0.6817, 0.6190])
d2 tensor([0.8283, 0.7510, 0.6817, 0.6190, 0.5639])
--------------------------------------------------------
The real label is     : tensor([1., 1., 1., 1., 1.])
The predicted label is: tensor([0.5922, 0.8503, 0.1884, 0.6923, 0.8164], grad_fn=<SigmoidBackward0>)
# # # # # # # # # # # # # # # # # # # # # # # # # # # #
contrastvie loss 0.002629
ranking loss     0.585141


In [158]:
print('one from TC, one from WT')
test_region(0,401,model, detail=detail)

one from TC, one from WT
# # # # # # # # # # # # # # # # # # # # # # # # # # # #
predicted distance    tensor([0.2544, 0.3216, 0.2279, 0.2392, 0.3349], grad_fn=<SqueezeBackward1>)
groundtruth distance  tensor([0.1717, 0.2490, 0.3183, 0.3810, 0.4361])
d1 tensor([1., 1., 1., 1., 1.])
d2 tensor([0.8283, 0.7510, 0.6817, 0.6190, 0.5639])
--------------------------------------------------------
The real label is     : tensor([1., 1., 1., 1., 1.])
The predicted label is: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SigmoidBackward0>)
# # # # # # # # # # # # # # # # # # # # # # # # # # # #
contrastvie loss 0.010127
ranking loss     0.000007


In [159]:
print('one from HT, one from WT')
test_region(200,401,model, detail=detail)

one from HT, one from WT
# # # # # # # # # # # # # # # # # # # # # # # # # # # #
predicted distance    tensor([0.7850, 0.6552, 0.8083, 0.7446, 0.5625], grad_fn=<SqueezeBackward1>)
groundtruth distance  tensor([0.8283, 0.7510, 0.6817, 0.6190, 0.5639])
d1 tensor([0., 0., 0., 0., 0.])
d2 tensor([0.8283, 0.7510, 0.6817, 0.6190, 0.5639])
--------------------------------------------------------
The real label is     : tensor([0., 0., 0., 0., 0.])
The predicted label is: tensor([0.0998, 0.1155, 0.0048, 0.0279, 0.0357], grad_fn=<SigmoidBackward0>)
# # # # # # # # # # # # # # # # # # # # # # # # # # # #
contrastvie loss 0.008575
ranking loss     0.059458


In [160]:
print('one from TC, one from HT')
test_region(0,200,model, detail=detail)

one from TC, one from HT
# # # # # # # # # # # # # # # # # # # # # # # # # # # #
predicted distance    tensor([0.9007, 1.0201, 1.0433, 1.0188, 0.9580], grad_fn=<SqueezeBackward1>)
groundtruth distance  tensor([1., 1., 1., 1., 1.])
d1 tensor([1., 1., 1., 1., 1.])
d2 tensor([0., 0., 0., 0., 0.])
--------------------------------------------------------
The real label is     : tensor([1., 1., 1., 1., 1.])
The predicted label is: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SigmoidBackward0>)
# # # # # # # # # # # # # # # # # # # # # # # # # # # #
contrastvie loss 0.002849
ranking loss     0.000000


In [161]:
print('Random sample two set of points')
g1 = np.random.randint(0,800,5)
g2 = np.random.randint(0,800,5)
test_region(g1,g2,model, detail=detail)

Random sample two set of points
# # # # # # # # # # # # # # # # # # # # # # # # # # # #
predicted distance    tensor([0.3047, 0.2810, 1.0206, 0.6131, 0.0650], grad_fn=<SqueezeBackward1>)
groundtruth distance  tensor([0.7203, 0.1224, 1.0000, 0.3893, 0.0000])
d1 tensor([0.0245, 1.0000, 1.0000, 0.0000, 1.0000])
d2 tensor([0.7448, 0.8776, 0.0000, 0.3893, 1.0000])
--------------------------------------------------------
The real label is     : tensor([0., 1., 1., 0., 0.])
The predicted label is: tensor([0.1767, 1.0000, 1.0000, 0.9997, 0.0029], grad_fn=<SigmoidBackward0>)
# # # # # # # # # # # # # # # # # # # # # # # # # # # #
contrastvie loss 0.050524
ranking loss     1.652789


In [110]:
id_ = 405

def probability_measurement(idx):
    g_x_test = all_g_embs[idx].repeat(200,1)
        
    g_1_test = all_g_embs[0:200]
    g_0_test = all_g_embs[200:400]
    image_batch_test = image_emb.repeat(200, 1, 1, 1, 1)

    with torch.no_grad():
        delta_x_0,_ = model(image_batch_test, g_x_test,g_0_test)
    with torch.no_grad():
        delta_x_1,_ = model(image_batch_test, g_x_test,g_1_test)
    prob = (torch.exp( - delta_x_1) / (torch.exp( - delta_x_1) + torch.exp( - delta_x_0))).mean()
    std =  (torch.exp( - delta_x_1) / (torch.exp( - delta_x_1) + torch.exp( - delta_x_0))).std()
    print('real d score {:.4f}'.format( all_scores[idx].item()))
    print('measured probabililty {:.4f}  STD {:.4f}'.format(prob.item(),std.item()) )

for idx in range(400,420,2):
    probability_measurement(idx)
    print()

real d score 0.9085
measured probabililty 0.6474  STD 0.0159

real d score 0.7510
measured probabililty 0.5923  STD 0.0147

real d score 0.6190
measured probabililty 0.6272  STD 0.0152

real d score 0.5146
measured probabililty 0.5887  STD 0.0125

real d score 0.4275
measured probabililty 0.5158  STD 0.0156

real d score 0.3563
measured probabililty 0.5290  STD 0.0117

real d score 0.2934
measured probabililty 0.5333  STD 0.0185

real d score 0.2404
measured probabililty 0.4947  STD 0.0188

real d score 0.1977
measured probabililty 0.4751  STD 0.0183

real d score 0.1630
measured probabililty 0.4551  STD 0.0265



In [108]:
import torch

def sample_pairs_no_diagonal(N, num_pairs, device='cpu'):
    """a
    Samples 'num_pairs' valid (i, j) with i != j from range(N).
    Returns tensors (i_idx, j_idx).
    """
    # 1. All indices from 0 to N*N - 1
    all_idx = torch.arange(N * N, device=device)

    # 2. Build a mask for diagonal entries (where i_idx == j_idx)
    #    i == j if floor_div == modulo
    mask_diagonal = (all_idx // N) == (all_idx % N)

    # 3. Filter out diagonal indices
    valid_idx = all_idx[~mask_diagonal]

    # 4. Randomly sample from valid_idx
    chosen = valid_idx[torch.randint(0, valid_idx.shape[0], (num_pairs,), device=device)]

    # 5. Convert back to (i_idx, j_idx)
    i_idx = chosen // N
    j_idx = chosen % N
    return i_idx, j_idx


# Usage Example
N = 512
num_pairs = 1024

i_idx, j_idx = sample_pairs_no_diagonal(N, num_pairs)
print("i_idx:", i_idx)
print("j_idx:", j_idx)
print("Any i == j?", (i_idx == j_idx).any().item())  # Should print 0 (False)


i_idx: tensor([107, 443, 421,  ...,  39, 291, 455])
j_idx: tensor([407,  16, 217,  ..., 411, 152, 185])
Any i == j? False
